@author Nassir Mohammad

# Preliminaries

In [3]:
import os
import sys 
sys.path.append('../')
sys.path.append('../scripts')

import warnings
from perception_nassir import Perception

import dataframe_image as dfi

import numpy as np
import pandas as pd

from scripts.utilities import apply_classifiers
from scripts.utilities import get_file_data

from sklearn.preprocessing import StandardScaler
from scripts.rendering_functions import highlight_max, highlight_min

image_save_path = ''
image_save_switch = False

# Paper 1 datasets 

> **Note on creditcard.csv**  
> The file `creditcard.csv` is not included in this repository under `data/ODDS_multivariate` due to its large size.  
> Download it from: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud and place it inside `data/ODDS_multivariate/`.  

In [4]:
# make table for dataset, # dimensions, # samples, # percentage of anomalies

base_path = "../data/ODDS_multivariate/"
data_properties_df = None

# loop over datasets in directory
for file_name in os.listdir(base_path):
 
    dataset_name = file_name.split('.')[0]
    file_path = base_path + file_name

    if dataset_name == "creditcard":
        df_temp = pd.read_csv(file_path, low_memory=False)
        X_original = df_temp.iloc[:, :-1].values.astype(float)
        y = df_temp.iloc[:, -1].values.astype(float)
    else:
        dataset_name, X_original, y = get_file_data(base_path, file_name)

    if dataset_name is None:
        continue

    # write dataset summary to dataframe
    data_properties_temp = pd.DataFrame({
        'Name': [dataset_name],
        '# examples': [X_original.shape[0]],
        '# features': [X_original.shape[1]],
        # '# anomalies': [y.sum()],
        '% anomalies': [round(y.sum() / X_original.shape[0] * 100, 2)],
    })

    data_properties_df = pd.concat(
        [data_properties_df, data_properties_temp]).reset_index(drop=True)
            

In [5]:
img_title = "Dataset properties"
path_save = image_save_path + "dataset_properties.png"

# order the dataset rows by name
data_properties_df = data_properties_df.sort_values(by=['Name']).reset_index(drop=True)

data_properties_df_styled = data_properties_df.style.format({'% anomalies': "{:.2f}"}).hide()

#data_properties_df_styled = data_properties_df.style.hide_index()

#dfi.export(data_properties_df,path_save)

data_properties_df_styled

Name,# examples,# features,% anomalies
cardio,1831,21,9.61
creditcard,284807,30,0.17
http,567498,3,0.39
musk,3062,166,3.17
satimage-2,5803,36,1.22
shuttle,49097,9,7.15
smtp,95156,3,0.03
thyroid,3772,6,2.47
wbc,378,30,5.56


In [6]:
# file names
# ########################
# file_name = "wbc.mat" 
# file_name = "cardio.mat"
# file_name = "thyroid.mat"
# file_name = "musk.mat"
# file_name = "shuttle.mat"
# file_name = "satimage-2.mat"
# file_name = "http.matv7"
# file_name = "smtp.matv7"
# file_name = "creditcard.csv"

classifiers = [
    'HBOS',  # to be ignored, first run in loop slower
    'HBOS',
    'IForest',
    'KNN',
    'LOF',
    'MCD',
    'OCSVM',
    'Perception',
]

metrics_df = None

with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    # loop over datasets in directory
    for file_name in os.listdir(base_path):

        dataset_name = file_name.split('.')[0]
        file_path = base_path + file_name

        if dataset_name == "creditcard":
            df_temp = pd.read_csv(file_path, low_memory=False)
            X_original = df_temp.iloc[:, :-1].values.astype(float)
            y = df_temp.iloc[:, -1].values.astype(float)

        else:
            dataset_name, X_original, y = get_file_data(base_path, file_name)

        if dataset_name is None:
            continue

        # scaling (very important to get right)
        # scale to zero mean and unit standard deviation along each feature
        sc = StandardScaler(with_mean=False)
        sc.fit(X_original)
        X = sc.transform(X_original)

        # Apply each classifier to dataset
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')

            print('current file in progress ...: {}'.format(dataset_name))

            metrics_temp = apply_classifiers(classifiers, dataset_name,
                                             predict_data=X,
                                             predict_labels=y,
                                             train_data=X)

        metrics_df = pd.concat([metrics_df, metrics_temp])

    metrics_df.reset_index(drop=True)

current file in progress ...: cardio
current classifier in progress: HBOS
total run time: 1.7232282508630306
current classifier in progress: HBOS
total run time: 0.004173791967332363
current classifier in progress: IForest
total run time: 0.08538341603707522
current classifier in progress: KNN
total run time: 0.3638605000451207
current classifier in progress: LOF
total run time: 0.4373539169318974
current classifier in progress: MCD
total run time: 0.2092928329948336
current classifier in progress: OCSVM
total run time: 0.2758453330025077
current classifier in progress: Perception
total run time: 0.004307040944695473
current file in progress ...: shuttle
current classifier in progress: HBOS
total run time: 0.04037479299586266
current classifier in progress: HBOS
total run time: 0.04018883313983679
current classifier in progress: IForest
total run time: 0.6890629169065505
current classifier in progress: KNN
total run time: 7.550494666909799
current classifier in progress: LOF
total run 

In [7]:
# create dataframe for precision
df_precision = metrics_df[['Dataset', 'Classifier', 'Precision']]
df_precision = pd.pivot_table(df_precision, values = 'Precision', index=['Classifier'], columns='Dataset').reset_index()
df_precision.columns.name = None

cols = [col for col in df_precision.columns]
formatdict = {}
for col in cols: formatdict[col] = "{:.3f}"
formatdict.pop('Classifier', None)

sub = df_precision.columns.values.tolist()
sub.remove('Classifier')
sub

df_precision = df_precision.style.hide().apply(highlight_max, subset=sub).format(formatdict)

# img_title = "Precision results"
# path_save = image_save_path + "dataset_precision.png"

# dfi.export(df_precision,path_save)

df_precision

Classifier,cardio,creditcard,http,musk,satimage-2,shuttle,smtp,thyroid,wbc
HBOS,0.443,0.015,0.051,0.316,0.114,0.689,0.003,0.202,0.421
IForest,0.470,0.016,0.039,0.316,0.119,0.705,0.002,0.235,0.395
KNN,0.350,nan,0.002,0.139,0.093,0.208,0.003,0.238,0.400
LOF,0.190,nan,0.001,0.137,0.040,0.116,0.003,0.060,0.400
MCD,0.462,0.015,0.039,0.316,0.122,0.689,0.002,0.235,0.368
OCSVM,0.503,nan,nan,0.316,0.120,0.697,0.002,0.214,0.395
Perception,0.591,0.031,0.149,0.890,0.333,0.907,0.006,0.247,0.483


In [8]:
# create dataframe for recall
df_recall = metrics_df[['Dataset', 'Classifier', 'Recall']]
df_recall = pd.pivot_table(df_recall, values = 'Recall', index=['Classifier'], columns='Dataset').reset_index()
df_recall.columns.name = None

cols = [col for col in df_recall.columns]
formatdict = {}
for col in cols: formatdict[col] = "{:.2f}"
formatdict.pop('Classifier', None)

sub = df_recall.columns.values.tolist()
sub.remove('Classifier')
sub

df_recall = df_recall.style.hide().apply(highlight_max, subset=sub).format(formatdict)

img_title = "Recall results"
path_save = image_save_path + "dataset_recall.png"

# dfi.export(df_recall,path_save)

df_recall

Classifier,cardio,creditcard,http,musk,satimage-2,shuttle,smtp,thyroid,wbc
HBOS,0.46,0.89,1.00,1.00,0.93,0.96,0.70,0.82,0.76
IForest,0.49,0.90,1.00,1.00,0.97,0.99,0.77,0.96,0.71
KNN,0.28,nan,0.04,0.27,0.66,0.22,0.73,0.87,0.67
LOF,0.16,nan,0.02,0.38,0.28,0.14,0.70,0.22,0.67
MCD,0.48,0.85,1.00,1.00,1.00,0.96,0.77,0.96,0.67
OCSVM,0.52,nan,nan,1.00,0.99,0.97,0.77,0.87,0.71
Perception,0.37,0.88,1.00,1.00,0.90,0.96,0.70,0.68,0.67


In [9]:
# create dataframe for F1-score
df_f1= metrics_df[['Dataset', 'Classifier', 'F1']]
df_f1 = pd.pivot_table(df_f1, values = 'F1', index=['Classifier'], columns='Dataset').reset_index()
df_f1.columns.name = None

cols = [col for col in df_f1.columns]
formatdict = {}
for col in cols: formatdict[col] = "{:.3f}"
formatdict.pop('Classifier', None)

sub = df_f1.columns.values.tolist()
sub.remove('Classifier')
sub

df_f1 = df_f1.style.hide().apply(highlight_max, subset=sub).format(formatdict)

img_title = "F1-score results"
path_save = image_save_path + "dataset_f1-score.png"

# dfi.export(df_f1,path_save)

df_f1

Classifier,cardio,creditcard,http,musk,satimage-2,shuttle,smtp,thyroid,wbc
HBOS,0.451,0.030,0.097,0.480,0.202,0.801,0.005,0.323,0.542
IForest,0.479,0.031,0.075,0.480,0.212,0.822,0.005,0.378,0.508
KNN,0.310,nan,0.004,0.183,0.163,0.214,0.005,0.374,0.500
LOF,0.176,nan,0.002,0.201,0.070,0.126,0.005,0.094,0.500
MCD,0.472,0.029,0.075,0.480,0.218,0.803,0.005,0.378,0.475
OCSVM,0.513,nan,nan,0.480,0.215,0.812,0.005,0.344,0.508
Perception,0.455,0.061,0.260,0.942,0.487,0.931,0.012,0.362,0.560


In [10]:
# create dataframe for Area under ROC curve
df1= metrics_df[['Dataset', 'Classifier', 'AUC']]
df1 = pd.pivot_table(df1, values = 'AUC', index=['Classifier'], columns='Dataset').reset_index()
df1.columns.name = None

cols = [col for col in df1.columns]
formatdict = {}
for col in cols: formatdict[col] = "{:.2f}"
formatdict.pop('Classifier', None)

sub = df1.columns.values.tolist()
sub.remove('Classifier')
sub

df1 = df1.style.hide().apply(highlight_max, subset=sub).format(formatdict)

img_title = "F1-score results"
path_save = image_save_path + "dataset_auc.png"

# dfi.export(df1,path_save)

df1

Classifier,cardio,creditcard,http,musk,satimage-2,shuttle,smtp,thyroid,wbc
HBOS,0.85,0.95,0.99,1.00,0.98,0.98,0.80,0.95,0.96
IForest,0.92,0.95,1.00,1.00,0.99,1.00,0.91,0.98,0.93
KNN,0.69,nan,0.25,0.62,0.93,0.63,0.91,0.96,0.95
LOF,0.55,nan,0.40,0.64,0.54,0.52,0.83,0.66,0.94
MCD,0.82,0.92,1.00,1.00,1.00,0.99,0.95,0.99,0.92
OCSVM,0.94,nan,nan,1.00,1.00,0.99,0.85,0.96,0.94
Perception,0.77,0.93,1.00,1.00,0.93,0.98,0.80,0.86,0.76


In [11]:
# create dataframe for total training and prediction time
df1= metrics_df[['Dataset', 'Classifier', 'Runtime']]
df1 = pd.pivot_table(df1, values = 'Runtime', index=['Classifier'], columns='Dataset').reset_index()
df1.columns.name = None

cols = [col for col in df1.columns]
formatdict = {}
for col in cols: formatdict[col] = "{:.4f}"
formatdict.pop('Classifier', None)

sub = df1.columns.values.tolist()
sub.remove('Classifier')
sub

df1 = df1.style.hide().apply(highlight_min, subset=sub).format(formatdict)

path_save = image_save_path + "dataset_total_time.png"

# dfi.export(df1,path_save)

df1

Classifier,cardio,creditcard,http,musk,satimage-2,shuttle,smtp,thyroid,wbc
HBOS,0.0042,0.7707,0.1343,0.0516,0.0218,0.0402,0.0256,0.0023,0.0025
IForest,0.0854,3.3035,6.5228,0.1163,0.1347,0.6891,1.2320,0.1025,0.0613
KNN,0.3639,nan,34.3683,2.9646,6.5649,7.5505,4.3572,0.2602,0.0328
LOF,0.4374,nan,17.6042,3.3173,17.6442,11.7318,1.6655,0.1555,0.0105
MCD,0.2093,14.4141,13.7766,10.7900,0.9988,7.6234,5.0330,0.3177,0.0395
OCSVM,0.2758,nan,nan,1.4091,3.5016,152.7382,555.4173,0.8173,0.0158
Perception,0.0043,0.2593,0.3245,0.0099,0.0055,0.0338,0.0579,0.0026,0.0011
